In [1]:
import pandas as pd
import networkx as nx
from xml.etree import ElementTree as ET
from tqdm import tqdm

In [2]:
df_cg = pd.read_csv('../edges/chemgene.csv',na_values=["N/A", "", "nan", "NaN", "NULL"],keep_default_na=False)
df_cd = pd.read_csv('../edges/chemdis.csv')
df_gd = pd.read_csv('../edges/genedis.csv',na_values=[ "N/A", "", "nan", "NaN", "NULL"], keep_default_na=False)

df_c = pd.read_csv('../vocab/CTD_chemicals.csv')
df_d = pd.read_csv('../vocab/CTD_diseases.csv')
df_g = pd.read_csv('../vocab/CTD_genes.csv',na_values=["N/A", "", "nan", "NaN", "NULL"],keep_default_na=False)

df_cg = df_cg[(df_cg['OrganismID'] == 9606)].drop_duplicates(subset=['ChemicalID', 'GeneSymbol'])
df_cd = df_cd.drop_duplicates(subset=['ChemicalID', 'DiseaseID'])  
df_gd = df_gd.drop_duplicates(subset=['GeneSymbol', 'DiseaseID'])

In [3]:
print(df_cg.info())
print(df_cd.info())
print(df_gd.info())

<class 'pandas.core.frame.DataFrame'>
Index: 812241 entries, 0 to 2993539
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ChemicalName        812241 non-null  object 
 1   ChemicalID          812241 non-null  object 
 2   CasRN               557625 non-null  object 
 3   GeneSymbol          812241 non-null  object 
 4   GeneID              812241 non-null  int64  
 5   GeneForms           809771 non-null  object 
 6   Organism            812241 non-null  object 
 7   OrganismID          812241 non-null  float64
 8   Interaction         812241 non-null  object 
 9   InteractionActions  812241 non-null  object 
 10  PubMedIDs           812241 non-null  object 
dtypes: float64(1), int64(1), object(9)
memory usage: 74.4+ MB
None
<class 'pandas.core.frame.DataFrame'>
Index: 3467034 entries, 0 to 9515672
Data columns (total 10 columns):
 #   Column               Dtype  
---  ------               -----  

In [4]:
known = pd.read_csv('chemsmiles.csv')
known.head()

,meshID,cids,smiles
0,C070055,35823,C1=CC(=C(C=C1C2=CC(=C(C=C2Cl)Cl)Cl)Cl)Cl
1,C024713,12389,CCCCCCCCCCCCCC
2,D000077612,166548,CCCCCOC1=CC=C(C=C1)C2=CC=C(C=C2)C3=CC=C(C=C3)C...
3,C538809,131634859,B(=O)OB1OB2OB(OB(O2)O1)[O-].[Na+]
4,C000626479,7565,C1=CC=C(C=C1)OC2=CC=C(C=C2)Br


In [5]:
df_cg = df_cg[df_cg["ChemicalID"].isin(known["meshID"])]
df_cd = df_cd[df_cd["ChemicalID"].isin(known["meshID"])]
df_cd['DiseaseID'] = df_cd['DiseaseID'].str[5:]
df_gd['DiseaseID'] = df_gd['DiseaseID'].str[5:]
df_d['DiseaseID'] = df_d['DiseaseID'].str[5:]

print(df_cg.info())
print(df_cd.info())

<class 'pandas.core.frame.DataFrame'>
Index: 703832 entries, 0 to 2993539
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ChemicalName        703832 non-null  object 
 1   ChemicalID          703832 non-null  object 
 2   CasRN               551580 non-null  object 
 3   GeneSymbol          703832 non-null  object 
 4   GeneID              703832 non-null  int64  
 5   GeneForms           701695 non-null  object 
 6   Organism            703832 non-null  object 
 7   OrganismID          703832 non-null  float64
 8   Interaction         703832 non-null  object 
 9   InteractionActions  703832 non-null  object 
 10  PubMedIDs           703832 non-null  object 
dtypes: float64(1), int64(1), object(9)
memory usage: 64.4+ MB
None
<class 'pandas.core.frame.DataFrame'>
Index: 2911974 entries, 1 to 9515672
Data columns (total 10 columns):
 #   Column               Dtype  
---  ------               -----  

In [6]:
chemicals = (set(df_cg['ChemicalID'].unique()) | set(df_cd['ChemicalID'].unique())) & set(known['meshID'])
genes = set(df_cg['GeneSymbol'].unique()) | set(df_gd['GeneSymbol'].unique())
diseases = set(df_cd['DiseaseID'].unique()) | set(df_gd['DiseaseID'].unique())

print(len(chemicals))
print(len(genes))
print(diseases)
print(len(chemicals)+len(genes)+len(diseases))

13943
28151
{'D020955', '300966', 'D001416', 'D016780', 'C535880', 'C535477', '615516', '614583', 'D053713', 'C537901', 'D005327', 'C567714', 'D013290', 'D016919', 'D000086722', 'C536339', '613717', '610536', 'C567101', 'D015818', 'D007972', 'D007669', 'C535598', '614487', 'D017380', 'C562999', 'C564326', 'D003389', 'D006938', 'D003424', 'C535325', 'D007973', 'C536164', '300863', '617409', 'D002418', 'C535387', '615871', '614563', 'C564474', 'D010022', 'D000086002', 'C536390', 'D028921', '614856', '615802', 'C535764', '615599', 'D009304', '300968', '136570', 'C563324', '300835', '613820', 'D018496', 'D020207', 'C536984', 'D006314', 'D006423', '617280', 'D011371', '615184', 'C564747', 'D006479', 'D020237', 'D006058', 'D010244', 'C535970', 'C562657', 'D009410', '125850', 'D049290', 'C537034', 'D001759', 'D006349', 'D001724', 'D055964', 'D058739', 'D007299', 'D000007', '615485', 'D016464', 'C538275', 'D001478', 'C536632', 'D011469', 'D031901', 'D004760', 'C535833', '613559', 'D005747', 'D

In [108]:
gene_emb = pd.read_csv('gtex_gene_tissue_expression.csv',na_values=["N/A", "", "nan", "NaN", "NULL"],keep_default_na=False)
gene_emb.head()

,EnsemblID,GeneSymbol,Whole Blood,Brain - Frontal Cortex (BA9),Brain - Cerebellar Hemisphere,Brain - Substantia nigra,Brain - Anterior cingulate cortex (BA24),Brain - Amygdala,Brain - Caudate (basal ganglia),Brain - Nucleus accumbens (basal ganglia),...,Liver - Hepatocyte,Pancreas - Mixed Cell,Pancreas - Islets,Pancreas - Acini,Liver - Portal Tract,Kidney - Medulla,Bladder,Cervix - Ectocervix,Cervix - Endocervix,Fallopian Tube
0,ENSG00000290825.2,DDX11L16,0.028955,0.018874,0.025773,0.017392,0.016015,0.018953,0.018269,0.021843,...,0.008585,0.002401,0.000000,0.005390,0.000000,0.009624,0.007607,0.009229,0.010787,0.013035
1,ENSG00000223972.6,DDX11L1,0.003192,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,ENSG00000310526.1,WASH7P,1.731737,0.944307,2.216555,0.795484,0.734542,0.681277,0.792692,0.787007,...,1.201638,1.357288,1.471623,1.541402,2.187545,1.714173,2.085198,2.291385,2.466782,2.514128
3,ENSG00000243485.6,MIR1302-2HG,0.011722,0.014546,0.023820,0.012514,0.010933,0.015694,0.012173,0.015497,...,0.026468,0.003592,0.000000,0.000000,0.000000,0.007528,0.021197,0.011385,0.026086,0.019993
4,ENSG00000237613.3,FAM138A,0.013435,0.026206,0.021936,0.024687,0.023050,0.021046,0.023017,0.027591,...,0.016027,0.000000,0.000000,0.005648,0.000000,0.010944,0.011410,0.018919,0.008539,0.023073


In [109]:
print(genes)
gene_emb = gene_emb[gene_emb["GeneSymbol"].isin(genes)]
gene_emb.info()

{'CIB2', 'LARP', 'C19ORF12', 'FAM21FP', 'SNX18P3', 'NXT2', 'GUSBP11', 'NRK', 'LIMD2', 'F13A1', 'ATXN7L1', 'C20ORF141', 'PCOLCE-AS1', 'PIGG', 'MIR3689D2', 'BRD1', 'CCDC9B', 'TTC31', 'SHC1', 'GH2', 'TAAR5', 'RIMS3', 'ESD', 'USP18', 'BICD1', 'TOPORS', 'OR5H2', 'VAMP1', 'FRMD5', 'BPIFA3', 'MTNAP1', 'ZNF414', 'FHL1P1', 'TCEAL1', 'NUDT16', 'ARAP1', 'SEC22C', 'CLEC3B', 'ZNF793', 'OSGEPL1-AS1', 'NXF5', 'RNU4-36P', 'PK', 'MESD', 'KRT18P30', 'WLS', 'TRAM1L1', 'ERVH-4', 'TMEM67', 'OR8D1', 'SNORA36C', 'PPHLN1', 'PSAPL1', 'RPLP1P4', 'MIR6739', 'PBP', 'TRABD2A', 'POLR3E', 'RPL10P7', 'ULBP2', 'ITGB3BP', 'GMIP', 'MS4A14', 'TRAC', 'KRTAP10-5', 'ALG1L6P', 'AFF1-AS1', 'UMPS', 'TUBB1', 'TPO', 'OTUD3', 'MYCBP', 'NKRF', 'CENPX', 'GRID2', 'STEAP3', 'FCAMR', 'FUOM', 'C12ORF71', 'CFAP144P1', 'YPEL4', 'SNORD111', 'IFT81', 'HIST1H2AL', 'SENP5', 'HACD3', 'WRNIP1', 'FAM224B', 'LITAF', 'E2F3', 'FALEC', 'NAP1L4', 'RAD51-AS1', 'DSC2', 'PRY2', 'PTTG3P', 'CIART', 'KRT222', 'TMEM246', 'CENPBD1P', 'MIR16-1', 'ZNF398', 'R

In [110]:
x = gene_emb.loc[gene_emb['GeneSymbol'] == 'CIB2'].values.flatten().tolist()[2:]
print(len(x))

68


In [7]:
G = nx.Graph()

for chem in chemicals:
    G.add_node(chem, node_type='chemical')
for gene in genes:
    G.add_node(gene, node_type='gene')
for disease in diseases:
    G.add_node(disease, node_type='disease')

for _, row in df_cg.iterrows():
    G.add_edge(row['ChemicalID'], row['GeneSymbol'], edge_type='chem_gene')
for _, row in df_cd.iterrows():
    G.add_edge(row['ChemicalID'], row['DiseaseID'], edge_type='chem_disease')
for _, row in df_gd.iterrows():
    G.add_edge(row['GeneSymbol'], row['DiseaseID'], edge_type='gene_disease')


In [8]:
import numpy as np

np.set_printoptions(linewidth=10**9)

with open("edges.txt", "w") as f:
    for g in G.edges(data=True):
        f.write(str(g) + "\n")

In [113]:

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(G.nodes(data=True))

Graph built: 49333 nodes, 3650027 edges
[('C023710', {'node_type': 'chemical'}), ('C432307', {'node_type': 'chemical'}), ('C447031', {'node_type': 'chemical'}), ('C041260', {'node_type': 'chemical'}), ('C497449', {'node_type': 'chemical'}), ('C109640', {'node_type': 'chemical'}), ('C024713', {'node_type': 'chemical'}), ('C022050', {'node_type': 'chemical'}), ('C031538', {'node_type': 'chemical'}), ('C045575', {'node_type': 'chemical'}), ('C098393', {'node_type': 'chemical'}), ('C457999', {'node_type': 'chemical'}), ('C000722507', {'node_type': 'chemical'}), ('D002336', {'node_type': 'chemical'}), ('C575077', {'node_type': 'chemical'}), ('C041594', {'node_type': 'chemical'}), ('C032642', {'node_type': 'chemical'}), ('C045951', {'node_type': 'chemical'}), ('C000626905', {'node_type': 'chemical'}), ('D000077608', {'node_type': 'chemical'}), ('C025280', {'node_type': 'chemical'}), ('C525424', {'node_type': 'chemical'}), ('D004157', {'node_type': 'chemical'}), ('D000069349', {'node_type': '

In [114]:
# df_c = df_c[['ChemicalID', 'ChemicalName', 'PubChemCID']].drop_duplicates(subset=['ChemicalID']).set_index('ChemicalID')
# df_d = df_d[['DiseaseID', 'DiseaseName', 'DiseaseSemanticType']].drop_duplicates(subset=['DiseaseID']).set_index('DiseaseID')
# df_g = df_g[['GeneSymbol', 'GeneSymbol', 'GeneName']].drop_duplicates(subset=['GeneSymbol']).set_index('GeneSymbol')


print(df_c.info())
print(df_g.info())
print(df_d.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179408 entries, 0 to 179407
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   ChemicalName       179408 non-null  object
 1   ChemicalID         179408 non-null  object
 2   CasRN              56623 non-null   object
 3   Definition         6190 non-null    object
 4   ParentIDs          179407 non-null  object
 5   TreeNumbers        179408 non-null  object
 6   ParentTreeNumbers  179407 non-null  object
 7   Synonyms           116886 non-null  object
dtypes: object(8)
memory usage: 11.0+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 641489 entries, 0 to 641488
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   GeneSymbol   641489 non-null  object
 1   GeneName     639154 non-null  object
 2   GeneID       641489 non-null  int64 
 3   AltGeneIDs   48428 non-null   object
 4

In [115]:
df_g = df_g[df_g["GeneSymbol"].isin(genes)]
df_g.info()


<class 'pandas.core.frame.DataFrame'>
Index: 28150 entries, 15907 to 641485
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   GeneSymbol   28150 non-null  object
 1   GeneName     28097 non-null  object
 2   GeneID       28150 non-null  int64 
 3   AltGeneIDs   20826 non-null  object
 4   Synonyms     26174 non-null  object
 5   BioGRIDIDs   20212 non-null  object
 6   PharmGKBIDs  21696 non-null  object
 7   UniProtIDs   20417 non-null  object
dtypes: int64(1), object(7)
memory usage: 1.9+ MB


In [116]:
paths = pd.read_csv('CTD_pathways.csv')
paths.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2567 entries, 0 to 2566
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   PathwayName  2567 non-null   object
 1   PathwayID    2567 non-null   object
dtypes: object(2)
memory usage: 40.2+ KB


In [117]:
dpath = pd.read_csv('../vocab/dispath.csv')
cpath = pd.read_csv('../vocab/chempath.csv')
gpath = pd.read_csv('../vocab/genepath.csv')

dpath['DiseaseID'] = dpath['DiseaseID'].str[5:]

cpath = cpath.drop_duplicates(subset=['ChemicalID', 'PathwayID'])  
gpath = gpath.drop_duplicates(subset=['GeneSymbol', 'PathwayID'])
dpath = dpath.drop_duplicates(subset=['DiseaseID', 'PathwayID'])

print(dpath.head())
print(diseases)

cpath = cpath[cpath["ChemicalID"].isin(chemicals)]
gpath = gpath[gpath["GeneSymbol"].isin(genes)]
dpath = dpath[dpath["DiseaseID"].isin(diseases)]

print(dpath.info())
print(cpath.info())
print(gpath.info())

                                  DiseaseName DiseaseID  \
0  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
1  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
2  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
3  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
4  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   

                                         PathwayName            PathwayID  \
0                              Androgen biosynthesis   REACT:R-HSA-193048   
1  Fatty acid, triacylglycerol, and ketone body m...   REACT:R-HSA-535734   
2                        Fatty Acyl-CoA Biosynthesis    REACT:R-HSA-75105   
3                                 Metabolic pathways        KEGG:hsa01100   
4                                         Metabolism  REACT:R-HSA-1430728   

  InferenceGeneSymbol  
0             HSD17B3  
1             HSD17B3  
2             HSD17B3  
3             HSD17B3  
4             HSD17B3  
{'D008310', 'C564948', 'C538275', 'C53

In [118]:
pathways = set(dpath['PathwayID'].unique()) | set(cpath['PathwayID'].unique())  | set(gpath['PathwayID'].unique())
len(pathways)

2363

In [119]:
# Group PathwayIDs by DiseaseID
dpaths = dpath.groupby("DiseaseID")["PathwayID"].apply(list).reset_index()
dpaths = dpaths.rename(columns={"PathwayID": "PathwayIDs"})
dpaths.set_index('DiseaseID',inplace=True)
print(dpaths)
# Convert to list of dictionaries
result_dp = dpaths.to_dict(orient="records")



                                                  PathwayIDs
DiseaseID                                                   
102200     [REACT:R-HSA-8937144, REACT:R-HSA-211859, REAC...
105500     [REACT:R-HSA-983712, KEGG:hsa04978, KEGG:hsa04...
106190     [REACT:R-HSA-1280218, KEGG:hsa04925, KEGG:hsa0...
108120     [KEGG:hsa04261, KEGG:hsa04260, KEGG:hsa05414, ...
108420     [REACT:R-HSA-1640170, REACT:R-HSA-1500620, REA...
...                                                      ...
D066087    [REACT:R-HSA-1296052, REACT:R-HSA-418457, KEGG...
D066088    [REACT:R-HSA-74160, REACT:R-HSA-983712, REACT:...
D066126    [REACT:R-HSA-166054, REACT:R-HSA-5617472, REAC...
D066253    [KEGG:hsa05412, REACT:R-HSA-380108, KEGG:hsa04...
D066263    [KEGG:hsa05014, REACT:R-HSA-2262752, REACT:R-H...

[5051 rows x 1 columns]


In [120]:
# Group PathwayIDs by DiseaseID
gpaths = gpath.groupby("GeneSymbol")["PathwayID"].apply(list).reset_index()
gpaths = gpaths.rename(columns={"PathwayID": "PathwayIDs"})
gpaths.set_index('GeneSymbol',inplace=True)

# Convert to list of dictionaries
result_gp = gpaths.to_dict(orient="records")

print(len(result_gp))

11458


In [121]:
# Group PathwayIDs by DiseaseID
cpaths = cpath.groupby("ChemicalID")["PathwayID"].apply(list).reset_index()
cpaths = cpaths.rename(columns={"PathwayID": "PathwayIDs"})
cpaths.set_index('ChemicalID',inplace=True)
print(cpaths)
# Convert to list of dictionaries
result_cp = cpaths.to_dict(orient="records")

print(len(result_cp))

                                                   PathwayIDs
ChemicalID                                                   
C000015     [REACT:R-HSA-382556, KEGG:hsa02010, KEGG:hsa04...
C000050     [REACT:R-HSA-1368108, REACT:R-HSA-156590, REAC...
C000081     [REACT:R-HSA-442660, REACT:R-HSA-112316, REACT...
C000121     [REACT:R-HSA-166054, REACT:R-HSA-450341, KEGG:...
C000154     [REACT:R-HSA-111447, REACT:R-HSA-111459, REACT...
...                                                       ...
D064753                  [KEGG:hsa05418, REACT:R-HSA-6785807]
D064791     [REACT:R-HSA-166054, KEGG:hsa04920, KEGG:hsa05...
D065146     [REACT:R-HSA-450341, KEGG:hsa05221, KEGG:hsa04...
D065366     [REACT:R-HSA-166054, REACT:R-HSA-450341, KEGG:...
D065819     [REACT:R-HSA-2426168, REACT:R-HSA-5423646, KEG...

[8529 rows x 1 columns]
8529


In [122]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs
import numpy as np

# Create generator once (faster)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=1024
)

def get_ecfp_embedding(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fp = morgan_gen.GetFingerprint(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


In [123]:
from sklearn.preprocessing import MultiLabelBinarizer
# suppose you have something like:
# chem_id | pathway
# C1      | p53
# C1      | MAPK
# C2      | Apoptosis

pths = list(set(paths['PathwayID'].unique()))
mlb_path = MultiLabelBinarizer(classes=pths)
mlb_path.fit([])
print((pths))

['REACT:R-HSA-187024', 'REACT:R-HSA-8875555', 'KEGG:hsa04392', 'REACT:R-HSA-168270', 'REACT:R-HSA-69202', 'KEGG:hsa04922', 'REACT:R-HSA-109582', 'REACT:R-HSA-2206292', 'KEGG:hsa_M00688', 'REACT:R-HSA-382551', 'REACT:R-HSA-167161', 'REACT:R-HSA-5658442', 'KEGG:hsa04915', 'REACT:R-HSA-844615', 'REACT:R-HSA-5693607', 'REACT:R-HSA-170145', 'KEGG:hsa_M00158', 'REACT:R-HSA-1483171', 'REACT:R-HSA-8948216', 'KEGG:hsa_M00397', 'REACT:R-HSA-156580', 'KEGG:hsa05150', 'REACT:R-HSA-375281', 'KEGG:hsa_M00160', 'REACT:R-HSA-1442490', 'REACT:R-HSA-111933', 'REACT:R-HSA-190861', 'REACT:R-HSA-2142691', 'REACT:R-HSA-3359471', 'REACT:R-HSA-1222449', 'REACT:R-HSA-69563', 'REACT:R-HSA-177162', 'REACT:R-HSA-1474228', 'REACT:R-HSA-5693579', 'REACT:R-HSA-157858', 'REACT:R-HSA-157579', 'REACT:R-HSA-6804116', 'REACT:R-HSA-194315', 'REACT:R-HSA-5602636', 'REACT:R-HSA-6781827', 'KEGG:hsa_M00043', 'REACT:R-HSA-189483', 'REACT:R-HSA-176409', 'REACT:R-HSA-1855215', 'REACT:R-HSA-1299344', 'REACT:R-HSA-77595', 'REACT:R

In [124]:
disease_slim = (
    df_d
    .groupby("DiseaseID")["SlimMappings"]
    .apply(lambda x: sorted(set(
        i for s in x.dropna() for i in s.split("|")
    )))
)
# slims = []
# for i in range(len(disease_slim)):
#     slims.extend(disease_slim.iloc[i])
# print(len(slims))
# print(len(set(slims)))
print(disease_slim)
mlb_slim = MultiLabelBinarizer()
disease_slim_emb = mlb_slim.fit_transform(disease_slim)
print(mlb_slim.classes_)
disease_slim_emb.shape

DiseaseID
100050     [Cardiovascular disease, Congenital abnormalit...
102200     [Cancer, Endocrine system disease, Nervous sys...
102510     [Congenital abnormality, Musculoskeletal disease]
102530                           [Urogenital disease (male)]
105500     [Mental disorder, Metabolic disease, Nervous s...
                                 ...                        
D066166    [Congenital abnormality, Connective tissue dis...
D066190         [Nervous system disease, Signs and symptoms]
D066229                                    [Mental disorder]
D066253    [Pathology (anatomical condition), Pathology (...
D066263                                [Pathology (process)]
Name: SlimMappings, Length: 13317, dtype: object
['Animal disease' 'Blood disease' 'Cancer' 'Cardiovascular disease' 'Congenital abnormality' 'Connective tissue disease' 'Digestive system disease' 'Drug-related side effect' 'Ear-nose-throat disease' 'Endocrine system disease' 'Environmental origin disorders' 'Eye dise

(13317, 38)

In [125]:
cpemb = mlb_path.transform(cpaths['PathwayIDs'])
gpemb = mlb_path.transform(gpaths['PathwayIDs'])
dpemb = mlb_path.transform(dpaths['PathwayIDs'])
print(cpemb.shape)

(8529, 2567)


In [126]:
def get_chemical_embedding(chem_id, smiles):
    ecfp = get_ecfp_embedding(smiles)
    try:
        pw = cpemb[cpaths.index.get_loc(chem_id)]
    except:
        pw = np.zeros(2567, dtype=np.float32)
    return np.concatenate([pw, ecfp]).astype(np.float32)


In [127]:
def get_disease_embedding(disease_id):
    try:
        pw = dpemb[dpaths.index.get_loc(disease_id)]
    except:
        pw = np.zeros(2567, dtype=np.float32)
    try:
        slim = disease_slim_emb[disease_slim.index.get_loc(disease_id)]
    except:
        slim = np.zeros(100, dtype=np.float32)
    
    
    return np.concatenate([pw, slim]).astype(np.float32)


In [132]:
def get_gene_embedding(gene_id):
    try:
        x = gene_emb.loc[gene_emb['GeneSymbol'] == gene_id].values.flatten().tolist()[2:70]
        if(len(x)<1):
            raise
    except:
        x = np.zeros(68, dtype=np.float32)
    try:
        pw = gpemb[gpaths.index.get_loc(gene_id)]
    except:
        pw = np.zeros(2567, dtype=np.float32)
    return np.concatenate([pw, x]).astype(np.float32)

In [133]:
from tqdm import tqdm
for disease_id in tqdm(diseases):
    G.nodes[disease_id]['x'] = get_disease_embedding(disease_id)
for gene_id in tqdm(genes):
    G.nodes[gene_id]['x'] = get_gene_embedding(gene_id)
for chem_id in tqdm(chemicals):
    G.nodes[chem_id]['x'] = get_chemical_embedding(chem_id,known.loc[known["meshID"] == chem_id, "smiles"].iloc[0])

 22%|██▏       | 3084/13943 [00:04<00:15, 686.63it/s][16:43:13] WARNING: not removing hydrogen atom without neighbors
[16:43:13] WARNING: not removing hydrogen atom without neighbors
[16:43:13] WARNING: not removing hydrogen atom without neighbors
[16:43:13] WARNING: not removing hydrogen atom without neighbors
 38%|███▊      | 5246/13943 [00:07<00:12, 669.83it/s][16:43:16] WARNING: not removing hydrogen atom without neighbors
[16:43:16] WARNING: not removing hydrogen atom without neighbors
 40%|████      | 5601/13943 [00:08<00:12, 686.53it/s][16:43:16] WARNING: not removing hydrogen atom without neighbors
[16:43:16] WARNING: not removing hydrogen atom without neighbors
 44%|████▍     | 6152/13943 [00:09<00:11, 675.91it/s][16:43:17] WARNING: not removing hydrogen atom without neighbors
[16:43:17] WARNING: not removing hydrogen atom without neighbors
[16:43:17] WARNING: not removing hydrogen atom without neighbors
 66%|██████▌   | 9219/13943 [00:14<00:06, 694.05it/s][16:43:22] WARNING: 

In [134]:
import pickle
with open("bio_graph_raw.pkl", "wb") as f:
    pickle.dump(G, f)


In [136]:
import numpy as np

np.set_printoptions(linewidth=10**9)

with open("graphnodes.txt", "w") as f:
    for g in G.nodes(data=True):
        f.write(str(g) + "\n")
